# Rigid motion: continuous Fourier workflow

This notebook uses the continuous form of `rigid_motion`. A callable pose with `epoch=` gives
two Fourier methods on `.at(targets)`: `fourier_series` for periodic motion and
`fourier_transform` for a finite window. It checks the periodic result against a closed form.

Requires Astropy and SciPy. Runs in a few seconds. Outputs are cleared in the saved file.

The guide is [Rigid motion: continuous Fourier analysis](../docs_site/docs/guide/rigid-motion-fourier.md).

In [ ]:
import numpy as np
from astropy import units as u
from scipy.spatial.transform import Rotation

from gragra.constants import G_SI
from gragra.geometry import Frame
from gragra.gravity_sources import PointMassCloud, rigid_motion
from gragra.targets import TargetPoints

EPOCH = (1_000_000_000, 0)  # GPS seconds, nanoseconds: the pose's time zero
world = Frame("world")
cloud = PointMassCloud(np.array([1.0]), np.array([[0.0, 0.0, 0.0]]))  # 1 kg at body origin
targets = TargetPoints(np.zeros((1, 3)) * u.m, ids=("centre",), frame=world)


def circular_pose(times):
    """Circle of radius 1 m and period 1 s; identity rotation."""
    t = times.to_value(u.s)
    angle = 2.0 * np.pi * t
    translation = np.stack([np.cos(angle), np.sin(angle), np.zeros_like(angle)], axis=1)
    return Rotation.identity(len(t)), translation * u.m


def passage_pose(times):
    """Straight passage c(t) = (t, 1, 0) m; identity rotation."""
    t = times.to_value(u.s)
    translation = np.stack([t, np.ones_like(t), np.zeros_like(t)], axis=1)
    return Rotation.identity(len(t)), translation * u.m


## Periodic motion: Fourier series

For the circle, the one-sided amplitudes are `A_1x = G` and `A_1y = -iG`. The other orders vanish.
The check uses a relative tolerance of `1e-6` in units of `G`.

In [ ]:
circle = rigid_motion(cloud, circular_pose, frame=world, epoch=EPOCH)
series = circle.at(targets, observables=("acceleration",)).fourier_series(
    period=1.0 * u.s,
    orders=[0, 1, 2],
    tref=EPOCH,
    tau_min=0.125 * u.s,
    atol=1e-9 * G_SI * u.m / u.s**2,
)
freqs = series.acceleration.frequencies.to_value(u.Hz)
ax = np.asarray(series.acceleration[0, 0])
ay = np.asarray(series.acceleration[0, 1])
print("frequencies (Hz):", freqs)
print("A_1x / G:", np.round(ax[1] / G_SI, 6))
print("A_1y / G:", np.round(ay[1] / G_SI, 6))

tol = 1e-6
assert abs(ax[1] / G_SI - 1.0) <= tol
assert abs(ay[1] / G_SI + 1j) <= tol
assert all(abs(ax[k]) <= tol * G_SI for k in (0, 2))
assert all(abs(ay[k]) <= tol * G_SI for k in (0, 2))
print("closed-form check passed")

## Finite window: Fourier transform

The transform integrates over `[t_start, t_stop]`. Its coefficients are one-sided and are not doubled.
The absolute tolerance has units of m/s, which is acceleration times seconds.

In [ ]:
passage = rigid_motion(cloud, passage_pose, frame=world, epoch=EPOCH)
spectrum = passage.at(targets, observables=("acceleration",)).fourier_transform(
    t_start=-2.0 * u.s,
    t_stop=2.0 * u.s,
    frequencies=np.array([0.0, 0.5, 1.0]) * u.Hz,
    tref=EPOCH,
    tau_min=0.5 * u.s,
    atol=1e-9 * G_SI * u.m / u.s,
    rtol=1e-10,
)
print("representation:", spectrum.context.representation)
print("X_x(f) [m/s]:", np.asarray(spectrum.acceleration[0, 0]))
print("observables:", spectrum.select("acceleration").observables)

## Notes

- `tref` is an explicit argument, so the phase reference is never the epoch by default.
- `memory_budget_bytes` is a per-lane budget for managed arrays. It is not a process memory limit.
- The gravitational constant is applied once, in the observable layer. The values above are not real-valued time-domain fields.